Install Libraries

In [ ]:
!pip install transformers
!pip install accelerate
!pip install -U bitsandbytes
!pip install datasets

Import Libraries

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from transformers import BitsAndBytesConfig
import json
import re

Load Model for Inference

In [ ]:
### Set Model Name and Huggingface Token Here
model_name = "meta-llama/Llama-3.1-8B"
model_name = "mistralai/Mistral-7B-v0.3"
model_name = "microsoft/phi-2"

access_token = "<HUGGINGFACE ACCESS TOKEN>"


# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)


# Load model and Tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=bnb_config,  trust_remote_code=True, use_auth_token=access_token)
print("model loaded")
print()

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=access_token)
tokenizer.pad_token = tokenizer.eos_token
print("tokenizer loaded")
print()

Load Dataset

In [ ]:
import json

# load the test/val set of any dataset for K-shot or MMR based K-shot inference
with open("/kaggle/input/rte-dataset/RTE_Tfidf_topk_test_set.json", "r") as f:
    test_set = json.load(f)
    # test_set = [json.loads(x) for x in f]
    
print(len(test_set))

In [ ]:
test_set[0]

In [ ]:
model

Zero Shot Inference

In [ ]:
# function to perform zero shot inference
def batch_zero_shot_inference(samples):
    prompts = []
    
    for sample in samples:
        sentence1 = sample["sentence1"]
        sentence2 = sample["sentence2"]
        
        # Refined prompt for clarity and precision
        prompt = f"""You are given two sentences. Classify the relationship between them. Respond with 'entailment' if the second sentence logically follows from the first, and 'not entailment' if it does NOT logically follow. Do not include any explanation or repetition in your answer.
First Sentence: "{sentence1}"
Second Sentence: "{sentence2}"
Answer:"""
        
        prompts.append(prompt)

    # Prepare inputs for the model
    model_inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda:0")
    
    # Generate output for all prompts in the batch
    outputs = model.generate(**model_inputs, max_new_tokens=5)  # Greedy decoding for deterministic output
    
    # Decode and return predictions
    predictions = [tokenizer.decode(output, skip_special_tokens=True).strip() for output in outputs]

    return predictions

# set config here
batch_size = 1
k = 0
model_name = 'SET MODEL NAME'
dataset_name = "SET DATASET NAME"


# Inference Loop
with open(f"/kaggle/working/{dataset_name}_test_set_ZS_pred_{model_name}.json", "w") as f:
    
    
    for i in range(0, len(test_set[:]), batch_size):

        # Get the current batch of samples
        batch_samples = test_set[i:i + batch_size]

        predicted_answers = batch_zero_shot_inference(batch_samples)
            

        for j, sample in enumerate(batch_samples):
            predicted_answer = predicted_answers[j]

            answers = re.findall(r'Answer:\s*(Entailment|entailment|Not Entailment|not entailment)', predicted_answer)
            predicted_answer = answers[-1].strip() if len(answers) == (k+1) else None

            if predicted_answer in ['Entailment', 'entailment']:
                predicted_answer = 1
            elif predicted_answer in ['Not Entailment', 'not entailment']:
                predicted_answer = 0
            else:
                predicted_answer = None

            temp = {
                'id': sample['idx'],
                'sentence1': sample['sentence1'],
                'sentence2': sample['sentence2'],
                'label': sample['label'],
                'predicted_label': predicted_answer
            }

            if count % 30 == 0:
                print(count)
                print(">>>>>>>>>>>>>{}".format(i+1), predicted_answer)
                print()

            json.dump(temp, f)
            f.write("\n")
            count += 1


K-shot Inference


In [ ]:
# function to perform few shot inference
def batch_k_shot_inference(samples, k, setting):
    label_map = {0: 'entailment', 1: 'not entailment'}
    prompts  = []
    for sample in samples:
        sentence1 = sample["sentence1"]
        sentence2 = sample["sentence2"]

        prompt = "You are given two sentences. Classify the relationship between them. Respond with 'entailment' if the second sentence logically follows from the first, and 'not entailment' if it does NOT logically follow. Do not include any explanation or repetition in your answer. Given below are some examples to guide your response. Here are a few examples:\n"

        for i in range(k):
            example_sentence1 = sample["reranked_top_k"][i]["sentence1"]
            example_sentence2 = sample["reranked_top_k"][i]["sentence2"]
            example_label = label_map[sample["reranked_top_k"][i]["label"]]
            prompt += f"Example {i + 1}:\nFirst Sentence: \"{example_sentence1}\"\nSecond Sentence: \"{example_sentence2}\"\nAnswer: {example_label}\n\n"

        prompt += (
            f"First Sentence: {sentence1}\n"
            f"Second Sentence: {sentence2}\n"
            f"Answer:"
        )

        prompts.append(prompt)
        
    model_inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda:0")
    outputs = model.generate(**model_inputs, max_new_tokens=5)
    predictions = [tokenizer.decode(output, skip_special_tokens=True).strip() for output in outputs]
    return predictions

# set config here
batch_size = 1
model_name = 'SET MODEL NAME'
dataset_name = "SET DATASET NAME"

### Set the config for the experiment you are performing (Few Shot / Few Shot MMR)
setting = 'few_shot'
# setting = 'few_shot_MMR'


# Inference Loop
for k in [1, 3, 5, 7, 9, 10]:
    with open(f"/kaggle/working/{dataset_name}_test_set_pred_k{k}_{model_name}.json", "w") as f:
      
        if setting == 'few_shot':
            setting_value = 'top_k'
        else:
            setting_value = 'reranked_top'
        
        
        for i in range(0, len(test_set[:]), batch_size):

            # Get the current batch of samples
            batch_samples = test_set[i:i + batch_size]

            predicted_answers = batch_k_shot_inference(batch_samples, k, setting=setting_value)
                

            for j, sample in enumerate(batch_samples):
                predicted_answer = predicted_answers[j]

                answers = re.findall(r'Answer:\s*(Entailment|entailment|Not Entailment|not entailment)', predicted_answer)
                predicted_answer = answers[-1].strip() if len(answers) == (k+1) else None

                if predicted_answer in ['Entailment', 'entailment']:
                    predicted_answer = 1
                elif predicted_answer in ['Not Entailment', 'not entailment']:
                    predicted_answer = 0
                else:
                    predicted_answer = None

                temp = {
                    'id': sample['idx'],
                    'sentence1': sample['sentence1'],
                    'sentence2': sample['sentence2'],
                    'label': sample['label'],
                    'predicted_label': predicted_answer
                }

                if count % 30 == 0:
                    print(count)
                    print(">>>>>>>>>>>>>{}".format(i+1), predicted_answer)
                    print()

                json.dump(temp, f)
                f.write("\n")
                count += 1